In [1]:
import os
import requests
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List
from agents import Agent, Runner, function_tool

load_dotenv()
print("Setup complete!")

Setup complete!


In [2]:
@function_tool
def tavily_search(query: str) -> str:
    """Search the web using Tavily and return results."""
    response = requests.post(
        "https://api.tavily.com/search",
        json={
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }
    )
    results = response.json().get("results", [])
    return "\n\n".join(
        f"**{r['title']}**\n{r['url']}\n{r['content']}"
        for r in results
    )
print("Tool created!")

Tool created!


In [3]:
class AnalysisOutput(BaseModel):
    key_trends: List[str]
    risks: List[str]
    insights: List[str]

class FinalReport(BaseModel):
    executive_summary: str
    markdown_report: str
    follow_up_questions: List[str]

print("Models defined!")

Models defined!


In [4]:
def create_researcher_agent():
    return Agent(
        name="Researcher",
        instructions=(
            "You are a research specialist. Use the tavily_search tool "
            "to gather information from multiple sources before summarizing. "
            "Always search first, then synthesize your findings."
        ),
        model="gpt-4o",
        tools=[tavily_search]
    )

analyst_agent = Agent(
    name="Analyst",
    instructions=(
        "You are a senior analyst. Given research findings, identify "
        "key trends, risks, and actionable insights. Be specific and "
        "data-driven."
    ),
    model="gpt-4o",
    output_type=AnalysisOutput
)

writer_agent = Agent(
    name="Writer",
    instructions=(
        "You are a professional report writer. Given analysis and insights, "
        "produce a polished executive summary, a detailed markdown report, "
        "and 3-5 follow-up questions for further research."
    ),
    model="gpt-4o",
    output_type=FinalReport
)
print("Agents created!")

Agents created!


In [5]:
async def run_pipeline(user_query: str, researcher_agent=None):
    if researcher_agent is None:
        researcher_agent = create_researcher_agent()

    print("Stage 1: Researching...")
    research_result = await Runner.run(researcher_agent, input=user_query)
    research_summary = research_result.final_output
    print("Research complete!\n")

    print("Stage 2: Analyzing...")
    analysis_result = await Runner.run(
        analyst_agent,
        input=f"Analyze these research findings:\n\n{research_summary}"
    )
    analysis = analysis_result.final_output
    print("Analysis complete!\n")

    print("Stage 3: Writing report...")
    writer_result = await Runner.run(
        writer_agent,
        input=f"Write a report based on this analysis:\n\n{analysis.model_dump_json()}"
    )
    print("Report complete!\n")
    return writer_result.final_output

print("Pipeline ready!")

Pipeline ready!


In [6]:
report = await run_pipeline(
    "What are the latest trends in AI agents and agentic workflows in 2026?"
)

print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(report.executive_summary)
print()
print("=" * 60)
print("FULL REPORT")
print("=" * 60)
print(report.markdown_report)
print()
print("=" * 60)
print("FOLLOW-UP QUESTIONS")
print("=" * 60)
for q in report.follow_up_questions:
    print(f"  - {q}")

Stage 1: Researching...
Research complete!

Stage 2: Analyzing...
Analysis complete!

Stage 3: Writing report...
Report complete!

EXECUTIVE SUMMARY
The analysis centers around the transformative impact of agentic AI systems on enterprise efficiency. Key trends include the adoption of autonomous AI for improved adaptability, multi-agent collaboration for enhanced problem-solving, and on-device AI for privacy and performance optimization. A pivotal insight is the integration of agentic AI with deterministic workflows to balance flexibility with precision. However, increased AI reliance raises concerns about system security, scalability, and operational complexity. Companies are advised to invest in multi-agent frameworks, prioritize on-device solutions, and ensure robust security measures to maximize potential while managing risks effectively.

FULL REPORT
# Analysis on Agentic AI Systems in Enterprises

## Key Trends
1. **Agentic AI Systems:** These systems significantly enhance enterp